# Checkpoint Analysis and Model Averaging

This notebook provides tools to:
1. Test the best/latest checkpoint model
2. Test individual checkpoints
3. Test combinations of checkpoints (model weight averaging)

## Model Averaging Methods
- **Simple Average**: Equal weights for all checkpoints
- **Weighted Average**: Different weights based on validation performance
- **Exponential Moving Average (EMA)**: Already implemented in training
- **Stochastic Weight Averaging (SWA)**: Average of checkpoints from different training stages

In [ ]:
# Import required libraries
import os
import glob
import re
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import json

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Import project modules
import sys
sys.path.append('/home/son/Desktop/GitHub/TinyRecursiveModels')

from pretrain import PretrainConfig, create_dataloader, create_model, load_checkpoint, average_checkpoints
from puzzle_dataset import PuzzleDatasetConfig, PuzzleDatasetMetadata
from utils.functions import load_model_class

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Configuration and Setup

In [ ]:
# Configuration
CHECKPOINT_DIR = "/home/son/Desktop/GitHub/TinyRecursiveModels/outputs/2026-01-31"  # Modify this to your checkpoint directory
CONFIG_FILE = None  # Will auto-detect from checkpoint directory

# Find all checkpoint directories
checkpoint_dirs = glob.glob(os.path.join(CHECKPOINT_DIR, "**/checkpoints"), recursive=True)
print(f"Found {len(checkpoint_dirs)} checkpoint directories:")
for i, cdir in enumerate(checkpoint_dirs):
    print(f"  {i}: {cdir}")

# Select a checkpoint directory
CHECKPOINT_PATH = checkpoint_dirs[0] if checkpoint_dirs else None
print(f"\nUsing: {CHECKPOINT_PATH}")

In [ ]:
# Helper functions
def find_all_checkpoints(checkpoint_dir: str) -> List[Tuple[int, str]]:
    """Find all checkpoint files and extract step numbers."""
    checkpoint_files = glob.glob(os.path.join(checkpoint_dir, "step_*"))
    # Filter out prediction files
    checkpoint_files = [f for f in checkpoint_files if not f.endswith(('.0', '.1', '.2', '.3'))]
    
    checkpoints = []
    for ckpt_file in checkpoint_files:
        match = re.search(r'step_(\d+)', ckpt_file)
        if match:
            step = int(match.group(1))
            checkpoints.append((step, ckpt_file))
    
    # Sort by step number
    checkpoints.sort(key=lambda x: x[0])
    return checkpoints

def load_config_from_checkpoint_dir(checkpoint_dir: str) -> dict:
    """Load configuration from the checkpoint directory."""
    config_files = glob.glob(os.path.join(os.path.dirname(checkpoint_dir), "*.yaml"))
    if config_files:
        import yaml
        with open(config_files[0], 'r') as f:
            return yaml.safe_load(f)
    return None

def print_checkpoint_info(checkpoints: List[Tuple[int, str]]):
    """Print information about available checkpoints."""
    print(f"\nFound {len(checkpoints)} checkpoints:")
    if checkpoints:
        print(f"  First checkpoint: Step {checkpoints[0][0]}")
        print(f"  Latest checkpoint: Step {checkpoints[-1][0]}")
        print(f"  Step interval: {checkpoints[1][0] - checkpoints[0][0] if len(checkpoints) > 1 else 'N/A'}")
        
        # Print first few and last few
        print("\n  First 5 checkpoints:")
        for step, path in checkpoints[:5]:
            print(f"    Step {step}: {os.path.basename(path)}")
        
        if len(checkpoints) > 10:
            print("  ...")
            print("  Last 5 checkpoints:")
            for step, path in checkpoints[-5:]:
                print(f"    Step {step}: {os.path.basename(path)}")

# Find and display checkpoints
if CHECKPOINT_PATH:
    all_checkpoints = find_all_checkpoints(CHECKPOINT_PATH)
    print_checkpoint_info(all_checkpoints)
else:
    print("No checkpoint directory found. Please set CHECKPOINT_DIR correctly.")

## 2. Load Model Configuration and Data

In [ ]:
# Load configuration
if CHECKPOINT_PATH:
    config_dict = load_config_from_checkpoint_dir(CHECKPOINT_PATH)
    if config_dict:
        print("Configuration loaded successfully!")
        print(f"\nKey configuration parameters:")
        print(f"  Data paths: {config_dict.get('data_paths', 'N/A')}")
        print(f"  Architecture: {config_dict.get('arch', {}).get('name', 'N/A')}")
        print(f"  Learning rate: {config_dict.get('lr', 'N/A')}")
        print(f"  Batch size: {config_dict.get('global_batch_size', 'N/A')}")
    else:
        print("Warning: Could not load configuration file. You'll need to specify it manually.")

In [ ]:
# Helper function to evaluate a model
def evaluate_model(model: nn.Module, 
                   test_loader: torch.utils.data.DataLoader,
                   eval_metadata: PuzzleDatasetMetadata,
                   evaluators: List,
                   device: str = 'cuda') -> Dict[str, float]:
    """Evaluate a model on test data."""
    model.eval()
    model.to(device)
    
    results = {}
    
    with torch.inference_mode():
        # Prepare evaluators
        return_keys = set()
        for evaluator in evaluators:
            evaluator.begin_eval()
            return_keys.update(evaluator.required_outputs)
        
        # Run evaluation
        set_ids = {k: idx for idx, k in enumerate(eval_metadata.sets)}
        metric_keys = None
        metric_values = None
        carry = None
        
        for set_name, batch, global_batch_size in tqdm(test_loader, desc="Evaluating"):
            # To device
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # Initialize carry
            with torch.device(device):
                carry = model.initial_carry(batch)
            
            # Forward pass (with multiple inference steps if needed)
            while True:
                carry, loss, metrics, preds, all_finish = model(
                    carry=carry, batch=batch, return_keys=return_keys
                )
                if all_finish:
                    break
            
            # Update evaluators
            for evaluator in evaluators:
                evaluator.update_batch(batch, preds)
            
            # Aggregate metrics
            set_id = set_ids[set_name]
            
            if metric_values is None:
                metric_keys = list(sorted(metrics.keys()))
                metric_values = torch.zeros(
                    (len(set_ids), len(metrics.values())), 
                    dtype=torch.float32, 
                    device=device
                )
            
            metric_values[set_id] += torch.stack([metrics[k] for k in metric_keys])
        
        # Process metrics
        if metric_values is not None:
            metric_values = metric_values.cpu().numpy()
            results = {
                set_name: {
                    metric_name: metric_values[set_id, metric_id]
                    for metric_id, metric_name in enumerate(metric_keys)
                }
                for set_id, set_name in enumerate(set_ids)
            }
            
            # Postprocess
            for set_name, m in results.items():
                count = m.pop("count", 1)
                results[set_name] = {k: v / count for k, v in m.items()}
        
        # Run evaluators
        for evaluator in evaluators:
            eval_results = evaluator.finish_eval()
            if eval_results:
                for set_name, metrics_dict in eval_results.items():
                    if set_name not in results:
                        results[set_name] = {}
                    results[set_name].update(metrics_dict)
    
    return results

## 3. Test Individual Checkpoints

In [ ]:
# Select checkpoints to test
# You can modify this to test specific checkpoints
CHECKPOINTS_TO_TEST = [
    all_checkpoints[-1],  # Latest checkpoint
    all_checkpoints[len(all_checkpoints)//2],  # Middle checkpoint
    # Add more checkpoints as needed
]

print("Checkpoints to test:")
for step, path in CHECKPOINTS_TO_TEST:
    print(f"  Step {step}: {path}")

In [ ]:
# Test individual checkpoints
# NOTE: This cell requires proper configuration setup
# You may need to adapt this based on your specific model and data setup

individual_results = {}

# This is a template - you'll need to properly initialize config, model, data loaders
# based on your specific setup
"""
for step, checkpoint_path in CHECKPOINTS_TO_TEST:
    print(f"\nTesting checkpoint at step {step}...")
    
    # Load model with checkpoint
    # You'll need to implement this based on your model architecture
    # model = load_model_with_checkpoint(checkpoint_path)
    
    # Evaluate
    # results = evaluate_model(model, test_loader, eval_metadata, evaluators)
    # individual_results[step] = results
    
    # print(f"Results: {results}")
"""

print("\n⚠️  To run this section, you need to:")
print("1. Load the proper configuration from your checkpoint directory")
print("2. Initialize the model architecture")
print("3. Create test data loaders")
print("4. Initialize evaluators")
print("\nSee the pretrain.py file for reference on how to set these up.")

## 4. Model Weight Averaging

In [ ]:
def load_checkpoint_weights(checkpoint_path: str) -> Dict[str, torch.Tensor]:
    """Load weights from a checkpoint file."""
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    # Handle different checkpoint formats
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        return checkpoint['model_state_dict']
    else:
        return checkpoint

def average_checkpoint_weights(checkpoint_paths: List[str], 
                              weights: Optional[List[float]] = None) -> Dict[str, torch.Tensor]:
    """Average weights from multiple checkpoints.
    
    Args:
        checkpoint_paths: List of paths to checkpoint files
        weights: Optional list of weights for weighted averaging. 
                If None, uses equal weights.
    
    Returns:
        Averaged state dict
    """
    if weights is None:
        weights = [1.0 / len(checkpoint_paths)] * len(checkpoint_paths)
    else:
        # Normalize weights
        total = sum(weights)
        weights = [w / total for w in weights]
    
    print(f"Averaging {len(checkpoint_paths)} checkpoints with weights: {weights}")
    
    # Load all checkpoints
    state_dicts = [load_checkpoint_weights(path) for path in checkpoint_paths]
    
    # Average weights
    averaged_state_dict = {}
    
    for key in state_dicts[0].keys():
        averaged_state_dict[key] = sum(
            w * sd[key].float() for w, sd in zip(weights, state_dicts)
        )
    
    return averaged_state_dict

def save_averaged_checkpoint(state_dict: Dict[str, torch.Tensor], 
                            output_path: str,
                            metadata: Optional[Dict] = None):
    """Save averaged checkpoint to disk."""
    checkpoint = {
        'model_state_dict': state_dict,
        'metadata': metadata or {}
    }
    torch.save(checkpoint, output_path)
    print(f"Saved averaged checkpoint to: {output_path}")

In [ ]:
# Example: Average the last N checkpoints
N_LAST_CHECKPOINTS = 5

if len(all_checkpoints) >= N_LAST_CHECKPOINTS:
    last_n_checkpoints = all_checkpoints[-N_LAST_CHECKPOINTS:]
    
    print(f"Averaging last {N_LAST_CHECKPOINTS} checkpoints:")
    for step, path in last_n_checkpoints:
        print(f"  Step {step}")
    
    # Simple average (equal weights)
    checkpoint_paths = [path for _, path in last_n_checkpoints]
    averaged_weights_equal = average_checkpoint_weights(checkpoint_paths)
    
    # Save averaged checkpoint
    output_path = os.path.join(CHECKPOINT_PATH, f"averaged_last_{N_LAST_CHECKPOINTS}_equal")
    save_averaged_checkpoint(
        averaged_weights_equal, 
        output_path,
        metadata={
            'averaged_checkpoints': [step for step, _ in last_n_checkpoints],
            'averaging_method': 'equal_weight',
            'n_checkpoints': N_LAST_CHECKPOINTS
        }
    )
else:
    print(f"Not enough checkpoints. Found {len(all_checkpoints)}, need at least {N_LAST_CHECKPOINTS}")

In [ ]:
# Example: Weighted average based on validation performance
# This requires you to have validation metrics for each checkpoint

# Dummy example - replace with actual validation accuracies
validation_accuracies = {
    # step: accuracy
    # e.g., 10000: 0.45, 20000: 0.47, etc.
}

if validation_accuracies:
    # Select checkpoints based on performance
    # E.g., top 5 performing checkpoints
    sorted_by_perf = sorted(validation_accuracies.items(), key=lambda x: x[1], reverse=True)
    top_k = 5
    top_checkpoints = sorted_by_perf[:top_k]
    
    print(f"Top {top_k} checkpoints by validation accuracy:")
    for step, acc in top_checkpoints:
        print(f"  Step {step}: {acc:.4f}")
    
    # Create weighted average based on performance
    checkpoint_paths = []
    weights = []
    
    for step, acc in top_checkpoints:
        # Find checkpoint path
        for ckpt_step, ckpt_path in all_checkpoints:
            if ckpt_step == step:
                checkpoint_paths.append(ckpt_path)
                weights.append(acc)  # Weight by accuracy
                break
    
    if checkpoint_paths:
        averaged_weights_performance = average_checkpoint_weights(checkpoint_paths, weights)
        
        output_path = os.path.join(CHECKPOINT_PATH, f"averaged_top_{top_k}_weighted")
        save_averaged_checkpoint(
            averaged_weights_performance,
            output_path,
            metadata={
                'averaged_checkpoints': [step for step, _ in top_checkpoints],
                'averaging_method': 'performance_weighted',
                'n_checkpoints': top_k,
                'weights': weights
            }
        )
else:
    print("No validation accuracies provided. Skipping weighted averaging.")

## 5. Stochastic Weight Averaging (SWA)

In [ ]:
# SWA: Average checkpoints from the last portion of training
# Typically uses checkpoints after learning rate has decayed

# Example: Average all checkpoints from the last 20% of training
swa_start_fraction = 0.8  # Start SWA at 80% of training

if all_checkpoints:
    total_steps = all_checkpoints[-1][0]
    swa_start_step = int(total_steps * swa_start_fraction)
    
    swa_checkpoints = [(step, path) for step, path in all_checkpoints if step >= swa_start_step]
    
    print(f"SWA: Averaging {len(swa_checkpoints)} checkpoints from step {swa_start_step} onwards")
    print(f"Checkpoint steps: {[step for step, _ in swa_checkpoints]}")
    
    if swa_checkpoints:
        checkpoint_paths = [path for _, path in swa_checkpoints]
        swa_weights = average_checkpoint_weights(checkpoint_paths)
        
        output_path = os.path.join(CHECKPOINT_PATH, "swa_averaged")
        save_averaged_checkpoint(
            swa_weights,
            output_path,
            metadata={
                'averaged_checkpoints': [step for step, _ in swa_checkpoints],
                'averaging_method': 'swa',
                'swa_start_step': swa_start_step,
                'n_checkpoints': len(swa_checkpoints)
            }
        )

## 6. Compare Different Averaging Strategies

In [ ]:
# Template for comparing different models
# You'll need to implement the evaluation logic

comparison_models = {
    'Latest Checkpoint': all_checkpoints[-1][1] if all_checkpoints else None,
    'Best Checkpoint': None,  # You need to determine this based on validation
    f'Average (Last {N_LAST_CHECKPOINTS})': os.path.join(CHECKPOINT_PATH, f"averaged_last_{N_LAST_CHECKPOINTS}_equal"),
    'SWA Average': os.path.join(CHECKPOINT_PATH, "swa_averaged"),
}

results_comparison = {}

print("Models to compare:")
for name, path in comparison_models.items():
    if path and os.path.exists(path):
        print(f"  ✓ {name}: {path}")
    else:
        print(f"  ✗ {name}: Not found")

# Evaluation template
"""
for model_name, checkpoint_path in comparison_models.items():
    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"\nEvaluating {model_name}...")
        
        # Load and evaluate model
        # model = load_model_with_checkpoint(checkpoint_path)
        # results = evaluate_model(model, test_loader, eval_metadata, evaluators)
        # results_comparison[model_name] = results
"""

In [ ]:
# Visualize comparison results
# This is a template - adapt based on your actual results structure

"""
if results_comparison:
    # Create comparison DataFrame
    comparison_data = []
    
    for model_name, results in results_comparison.items():
        for set_name, metrics in results.items():
            for metric_name, value in metrics.items():
                comparison_data.append({
                    'Model': model_name,
                    'Set': set_name,
                    'Metric': metric_name,
                    'Value': value
                })
    
    df_comparison = pd.DataFrame(comparison_data)
    
    # Plot comparison
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Bar plot for main accuracy metric
    accuracy_df = df_comparison[df_comparison['Metric'] == 'accuracy']
    sns.barplot(data=accuracy_df, x='Model', y='Value', hue='Set', ax=axes[0])
    axes[0].set_title('Accuracy Comparison')
    axes[0].set_ylabel('Accuracy')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Table view
    pivot_table = df_comparison.pivot_table(
        index='Model', 
        columns=['Set', 'Metric'], 
        values='Value'
    )
    
    axes[1].axis('tight')
    axes[1].axis('off')
    table = axes[1].table(
        cellText=pivot_table.values,
        rowLabels=pivot_table.index,
        colLabels=[f"{s}\n{m}" for s, m in pivot_table.columns],
        cellLoc='center',
        loc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    
    plt.tight_layout()
    plt.show()
    
    # Display summary table
    print("\nSummary Results:")
    print(pivot_table.to_string())
"""

print("Comparison visualization template ready.")
print("Uncomment and run after implementing evaluation.")

## 7. Advanced: Checkpoint Trajectory Analysis

In [ ]:
# Analyze training trajectory by evaluating multiple checkpoints
# This helps understand which checkpoints to average

def analyze_checkpoint_trajectory(checkpoints: List[Tuple[int, str]], 
                                 sample_every: int = 1) -> pd.DataFrame:
    """Analyze model performance across training trajectory.
    
    Args:
        checkpoints: List of (step, path) tuples
        sample_every: Sample every Nth checkpoint to reduce computation
    
    Returns:
        DataFrame with checkpoint analysis results
    """
    sampled_checkpoints = checkpoints[::sample_every]
    
    trajectory_data = []
    
    print(f"Analyzing {len(sampled_checkpoints)} checkpoints...")
    
    # Template - implement based on your evaluation setup
    """
    for step, checkpoint_path in tqdm(sampled_checkpoints):
        # Load and evaluate
        model = load_model_with_checkpoint(checkpoint_path)
        results = evaluate_model(model, test_loader, eval_metadata, evaluators)
        
        # Extract key metrics
        for set_name, metrics in results.items():
            trajectory_data.append({
                'step': step,
                'set': set_name,
                **metrics
            })
    """
    
    # return pd.DataFrame(trajectory_data)
    return None

print("Trajectory analysis function defined.")
print("This is computationally expensive - use sample_every to reduce cost.")

In [ ]:
# Visualize training trajectory
"""
if all_checkpoints:
    # Sample every 5th checkpoint to speed up analysis
    trajectory_df = analyze_checkpoint_trajectory(all_checkpoints, sample_every=5)
    
    if trajectory_df is not None:
        fig, axes = plt.subplots(2, 1, figsize=(14, 10))
        
        # Plot accuracy over time
        for set_name in trajectory_df['set'].unique():
            set_data = trajectory_df[trajectory_df['set'] == set_name]
            axes[0].plot(set_data['step'], set_data['accuracy'], 
                        marker='o', label=set_name)
        
        axes[0].set_xlabel('Training Step')
        axes[0].set_ylabel('Accuracy')
        axes[0].set_title('Model Accuracy Over Training')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Plot loss over time
        for set_name in trajectory_df['set'].unique():
            set_data = trajectory_df[trajectory_df['set'] == set_name]
            axes[1].plot(set_data['step'], set_data.get('loss', []), 
                        marker='o', label=set_name)
        
        axes[1].set_xlabel('Training Step')
        axes[1].set_ylabel('Loss')
        axes[1].set_title('Model Loss Over Training')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Identify best checkpoints
        best_checkpoints = trajectory_df.loc[
            trajectory_df.groupby('set')['accuracy'].idxmax()
        ]
        print("\nBest checkpoints by set:")
        print(best_checkpoints[['step', 'set', 'accuracy']])
"""

print("Trajectory visualization template ready.")

## 8. Export Best Model

In [ ]:
# After determining the best strategy, export the final model

def export_final_model(checkpoint_path: str, 
                      output_dir: str,
                      model_name: str = "final_model"):
    """Export final model with metadata."""
    os.makedirs(output_dir, exist_ok=True)
    
    # Copy checkpoint
    import shutil
    output_path = os.path.join(output_dir, f"{model_name}.pt")
    shutil.copy(checkpoint_path, output_path)
    
    # Save metadata
    metadata = {
        'source_checkpoint': checkpoint_path,
        'export_date': str(pd.Timestamp.now()),
        'model_name': model_name,
    }
    
    metadata_path = os.path.join(output_dir, f"{model_name}_metadata.json")
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"Exported model to: {output_path}")
    print(f"Metadata saved to: {metadata_path}")

# Example usage:
# export_final_model(
#     checkpoint_path=os.path.join(CHECKPOINT_PATH, "swa_averaged"),
#     output_dir="/path/to/export",
#     model_name="trm_arc_swa_best"
# )

## Summary and Recommendations

### Key Findings (to be filled after running experiments):
1. **Best Individual Checkpoint**: [To be determined]
2. **Best Averaging Strategy**: [To be determined]
3. **Performance Improvement from Averaging**: [To be determined]

### Recommended Strategies:

1. **For Maximum Accuracy**:
   - Test SWA (Stochastic Weight Averaging) from last 20% of training
   - Test average of top-5 checkpoints by validation accuracy
   - Use EMA model if available

2. **For Robustness**:
   - Average last 3-5 checkpoints with equal weights
   - Ensemble predictions from multiple averaged models

3. **For Fast Deployment**:
   - Use latest checkpoint if performance is stable
   - Otherwise use best validation checkpoint

### Next Steps:
1. Implement the evaluation functions with your specific model/data setup
2. Run experiments to compare strategies
3. Select and export the best model
4. Document findings and update this summary